# nb_04 — Lomb-Scargle periods

Goal (see `docs/SPEC_V01.md`, rough plan step 4): run a Lomb-Scargle periodogram per object,
per band, using each light curve's duration (from nb_02) to bound the search's longest
findable period. Measure execution time — the spec flags this explicitly, since periods are
one of the more expensive things this workshop asks for. Works on `dia_object_lc_hq` — the
whole HQ sample (`nDiaSources > 100`, ~399k objects) with nb_02's LC stats and nb_03's
per-band magnitudes already merged into it in place.
</cell id="intro">

In [ ]:
import shutil
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.timeseries import LombScargle, LombScargleMultiband

import lsdb
from datapaths import Datapaths

sys.path.insert(0, str(Path.cwd().parent / "src"))
from dataio import select_slice

from dask.distributed import Client

client = Client(n_workers=4, threads_per_worker=1, memory_limit="auto")

dp = Datapaths()
hq_path = dp["dp2_subset"] / "dia_object_lc_hq"
hq_cat = lsdb.open_catalog(hq_path)
print(hq_cat.npartitions, "partitions")
hq_cat

## 0. Pick a slice of the HQ sample to explore

Same pattern as nb_02/nb_03: `select_slice` (`src/dataio/hq_sample.py`) gets you either one
partition or a cone search's worth of `hq_cat` to look at directly, without touching the
whole ~399k-object sample the periods below are computed over.

In [ ]:
SLICE_MODE = "partition"  # "partition" or "cone_search"
PARTITION_INDEX = 200
CONE_RA, CONE_DEC, CONE_RADIUS_ARCSEC = 150.0, 2.0, 1800  # ~0.5 deg

slice_cat = select_slice(
    hq_cat,
    mode=SLICE_MODE,
    partition_index=PARTITION_INDEX,
    ra=CONE_RA,
    dec=CONE_DEC,
    radius_arcsec=CONE_RADIUS_ARCSEC,
)
slice_df = slice_cat.compute()
print(f"{SLICE_MODE}: {len(slice_df)} objects")
slice_df.drop(columns=["diaSource", "diaObjectForcedSource"]).head()

## 1. Per-band Lomb-Scargle via `map_partitions`

Same `map_partitions` pattern as nb_02/nb_03, this time running `astropy.timeseries.LombScargle`
on `diaSource.scienceMag` vs. `diaSource.midpointMjdTai`, per band (bands don't share a
cadence, so mixing them isn't meaningful — same reasoning as nb_02's amplitude stats). This
cell only defines the per-partition function — `map_partitions` is lazy, so nothing runs yet;
section 2 below is where `write_catalog` actually triggers the computation.

**Longest findable period, from nb_02's `duration_days`:** at least ~2 cycles need to fit in
the observed baseline to say anything about a period, so `maximum_period = duration_days / 2`,
passed to `LombScargle.autopower` as `minimum_frequency`. The *shortest* findable period is left
to `autopower`'s own default (`samples_per_peak=5`, `nyquist_factor=5`, both astropy defaults) —
nb-v01's per-object cadence was uneven enough that a fixed floor didn't seem worth adding on
top of the default.

**A real-data finding from nb-v01 that shapes the threshold below:** per-band point counts in
that 7,036-object subset were much thinner than `nDiaSources` (the *total* across all 6 bands)
suggests — median counts per band were `z`: 7, `i`: 3, `r`: 3, `g`: 2, `u`/`y`: 0. Only ~34% of
objects reached 10+ points in their best band at all. `MIN_POINTS_FOR_LS = 10` below is already
a compromise — lower and the periodogram is mostly noise, higher and there's barely anything
left to plot. Whether the same pattern holds at HQ scale (`nDiaSources > 100`, not nb-v01's
`> 10`) isn't checked yet — section 2's per-band coverage printout will show it.
</cell id="7f98941e">

### Heavy-calculation flag

Both the single-band (section 2) and multiband (section 5) Lomb-Scargle passes are gated by
the same `RUN_HEAVY_CALC` flag — leave it `False` (the default) to skip straight to reading
the columns `dia_object_lc_hq` already has in sections 3-4 and the multiband comparison,
without rerunning either periodogram pass. **First time populating this branch's registry:**
set it `True` once so those columns actually exist.

**Both passes write to a temporary path first, then promote it into `dia_object_lc_hq`** —
same pattern nb_02/nb_03 use. `dia_object_lc_hq` already has content by the time either of
these runs (nb_01's base, plus nb_02/nb_03's columns, plus — for section 5 — section 2's own
periods), so `resume=True` straight onto it can't be used directly: it only checks whether a
pixel's output file already exists, not whether it has the columns this particular run would
add, so it would silently skip writing new columns for every pixel that's already there.
Writing resumably to a temporary path and only swapping it in once that write actually
completes protects the expensive part (a crash partway only costs the temporary path's
not-yet-written partitions) while keeping the promote step itself a fast directory rename, not
a recompute.

Same caveat as nb_02/nb_03 either way: `resume=True` only knows which pixels exist in the
temporary path, not whether the underlying function changed — delete it first (or pass
`overwrite=True` for that write instead) for a clean rebuild after editing
`band_periods_partition` or `multiband_period_partition`.

**Not run at HQ scale in this session** (per-object logic only checked against a small slice
— see `docs/changelog.md`), but extrapolating nb-v01's measured per-object rates to the whole
~399k-object HQ sample: single-band (~0.56 ms/object) projects to **~3.7 minutes**; multiband
(~35.14 ms/object, ~59x single-band) projects to **~3.9 hours**. Both run under the same flag,
so flipping it on reruns both together — budget for the multiband pass specifically, e.g. run
it in the background rather than waiting on it interactively.

In [2]:
RUN_HEAVY_CALC = False

In [ ]:
BANDS = "ugrizy"
MIN_POINTS_FOR_LS = 10
LS_KWARGS = dict(samples_per_peak=5, nyquist_factor=5)


def band_periods_partition(df):
    out = {}
    for b in BANDS:
        out[f"{b}_period_days"] = np.full(len(df), np.nan)
        out[f"{b}_period_power"] = np.full(len(df), np.nan)
    out["best_period_days"] = np.full(len(df), np.nan)
    out["best_period_power"] = np.full(len(df), np.nan)

    for i in range(len(df)):
        row = df.iloc[i]
        t_all = np.asarray(row["diaSource"]["midpointMjdTai"], dtype=float)
        mag_all = np.asarray(row["diaSource"]["scienceMag"], dtype=float)
        band_all = np.asarray(row["diaSource"]["band"])

        duration = row["duration_days"]
        max_period = duration / 2 if duration and duration > 0 else np.nan

        best_period, best_power = np.nan, -np.inf
        for b in BANDS:
            sel = (band_all == b) & np.isfinite(mag_all) & np.isfinite(t_all)
            t, mag = t_all[sel], mag_all[sel]

            if len(t) < MIN_POINTS_FOR_LS or not np.isfinite(max_period) or max_period <= 0:
                continue

            freq, power = LombScargle(t, mag).autopower(minimum_frequency=1.0 / max_period, **LS_KWARGS)
            j = np.argmax(power)
            out[f"{b}_period_days"][i] = float(1.0 / freq[j])
            out[f"{b}_period_power"][i] = float(power[j])
            if power[j] > best_power:
                best_period, best_power = out[f"{b}_period_days"][i], out[f"{b}_period_power"][i]

        out["best_period_days"][i] = best_period
        out["best_period_power"][i] = best_power if np.isfinite(best_power) else np.nan

    for k, v in out.items():
        df[k] = v
    return df

## 2. Compute, write, and check execution time

Unlike section 1's function definition, this cell is where the computation actually happens —
`map_partitions` is lazy, so nothing runs until `write_catalog` (to a temporary path, with
`resume=True` — see the heavy-calculation-flag markdown above) triggers it, partition by
partition. Timing wraps that write call itself rather than a separate `.compute()`, since
that's what forces the work now.

The spec calls this out specifically: periods are expensive enough that performance needs
checking before the workshop, not during it. Uses the per-object rate for a rough
extrapolation to `dia_object_collection`'s full 232,004,216 rows (`_metadata` footer count, no
data read — same trick nb_01 used).

This is a naive linear projection, not a real estimate: the HQ sample is already filtered to
`nDiaSources > 100`, so most raw-collection objects have far fewer `diaSource` points and would
fail `MIN_POINTS_FOR_LS` almost instantly rather than running a full periodogram — meaning the
real full-collection number is very likely *better* than this projection, not worse. Treat it
as an order-of-magnitude ceiling for planning, not a promise.
</cell id="ed8197c4">

In [ ]:
FULL_COLLECTION_ROWS = 232_004_216

if RUN_HEAVY_CALC:
    tmp_path = hq_path.parent / (hq_path.name + "_nb04_singleband_tmp")

    with_periods_cat = hq_cat.map_partitions(band_periods_partition)

    t0 = time.time()
    with_periods_cat.write_catalog(tmp_path, resume=True, create_thumbnail=False, progress_bar=False)
    elapsed = time.time() - t0

    n_obj = hq_cat.hc_structure.catalog_info.total_rows
    print(f"{elapsed:.1f} s for {n_obj} objects ({1000 * elapsed / n_obj:.2f} ms/object)")
    projected_seconds = FULL_COLLECTION_ROWS * (elapsed / n_obj)
    print(f"naive linear projection for the full collection: {projected_seconds / 3600:.1f} hours, serial, single-machine")

    # Promote: the resumable write to a fresh path is complete, so swap it in for the old
    # collection at hq_path — a fast rename, not a recompute.
    if hq_path.exists():
        shutil.rmtree(hq_path)
    tmp_path.rename(hq_path)
    print("promoted", tmp_path, "->", hq_path)

    hq_cat = lsdb.open_catalog(hq_path)
    coverage_df = hq_cat[[f"{b}_period_days" for b in BANDS]].compute()
    for b in BANDS:
        print(f"  {b}: {coverage_df[f'{b}_period_days'].notna().sum()} periods")
    print("any band:", coverage_df.notna().any(axis=1).sum())

    dp.register(
        name="dia_object_lc_hq",
        type="dp2_subset",
        fmt="hats-collection",
        src_path=str(hq_path / "collection.properties"),
        tags=["nb01", "nb02", "nb03", "nb04", "dp2", "diaObject", "periods", "lomb-scargle", "hq"],
        notes=(
            "nb_01's base HQ sample plus nb_02's stats and nb_03's magnitudes, plus "
            "{band}_period_days/{band}_period_power (per band, Lomb-Scargle on "
            "diaSource.scienceMag, >=10 points required, max period = duration_days/2) and "
            "best_period_days/best_period_power (highest-power band), from nb_04."
        ),
        overwrite_history=True,
    )
    dp.print_paths(name="dia_object_lc_hq")
else:
    print(f"RUN_HEAVY_CALC=False — skipping write/register; {hq_path} already has these columns.")

## 3. Period histogram

Reopen the written collection, then look at `best_period_days` (whichever band had the
highest-power periodogram for that object) restricted to objects that got one at all.
`best_period_power` isn't a calibrated false-alarm probability — just this periodogram's raw
peak power — so this is "periods LS found," not "periods that are real."


In [ ]:
hq_cat = lsdb.open_catalog(hq_path)
print(sorted(hq_cat.columns))

periods_df = hq_cat[["diaObjectId", "best_period_days", "best_period_power"]].compute()
periods_df = periods_df.dropna(subset=["best_period_days"])
print(periods_df.shape)

fig, ax = plt.subplots(figsize=(7, 4.5))
bins = np.logspace(np.log10(periods_df["best_period_days"].min()), np.log10(periods_df["best_period_days"].max()), 40)
ax.hist(periods_df["best_period_days"], bins=bins, color="C0")
ax.set_xscale("log")
ax.set_xlabel("best_period_days")
ax.set_ylabel("objects")
ax.set_title(f"whole HQ sample (n={len(periods_df):,})")
fig.tight_layout()

## 4. Sanity check: fold the highest-power light curve

Take the object with the single highest `best_period_power` in the whole subset, fold its
best band's light curve on that period, and look at it — the same "spot-check, don't trust the
histogram blindly" instinct the spec asks for throughout.


In [ ]:
fold_cols = (
    ["diaObjectId", "diaSource", "best_period_days", "best_period_power"]
    + [f"{b}_period_power" for b in BANDS]
)
fold_df = hq_cat[fold_cols].compute()
fold_df = fold_df.dropna(subset=["best_period_days"])

top = fold_df.loc[fold_df["best_period_power"].idxmax()]
# pd.notna() first: per-band *_period_power is a nullable-float column, and `pd.NA == x` raises
# (ambiguous truth value) rather than returning False the way `np.nan == x` would.
best_band = next(
    b for b in BANDS
    if pd.notna(top[f"{b}_period_power"]) and float(top[f"{b}_period_power"]) == float(top["best_period_power"])
)
period = top["best_period_days"]

ds = top["diaSource"]
sel = ds["band"] == best_band
t = ds.loc[sel, "midpointMjdTai"].to_numpy()
mag = ds.loc[sel, "scienceMag"].to_numpy()
phase = (t / period) % 1.0

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(phase, mag, color=f"C{BANDS.index(best_band)}")
ax.scatter(phase + 1, mag, color=f"C{BANDS.index(best_band)}", alpha=0.3)  # second cycle, for continuity
ax.invert_yaxis()
ax.set_xlabel("phase")
ax.set_ylabel(f"{best_band} scienceMag")
ax.set_title(
    f"diaObjectId={top['diaObjectId']}, band={best_band}, "
    f"period={period:.3f} d, power={top['best_period_power']:.3f}, n={sel.sum()}"
)
fig.tight_layout()

## 5. Multiband Lomb-Scargle

Section 1's "Next"-list item, attempted: `astropy.timeseries.LombScargleMultiband` fits one
shared base period across *all* bands' epochs at once (plus small per-band offset/amplitude
terms — `nterms_base=1`, `nterms_band=1`, both astropy defaults), instead of picking a single
"best" band per object the way sections 1-4 do. Same `duration_days / 2` bound on the longest
findable period as before.

The payoff shows up immediately in coverage: pooling all bands means an object doesn't need
10+ points in any *one* band, just 10+ combined — which nearly every object in the HQ sample
satisfies by construction (`nDiaSources > 100` was nb_01's filter). Expect close to 100%
coverage here.

That coverage isn't free — timed the same way as section 2, this is markedly more expensive
per object than the single-band version (nb-v01 measured ~59x), worth knowing before pointing
it at anything bigger than the HQ sample.

**Same write-to-temp-then-promote pattern as section 2** (see the heavy-calculation-flag
markdown above for why `resume=True` can't be used directly against `dia_object_lc_hq` once it
already has content). `map_partitions` here operates on `hq_cat`, which already carries every
column nb_02/nb_03/section 2 added — so the temporary collection carries everything too, not
just the two new multiband columns. Once its resumable write actually finishes, swapping it in
for the old collection is a fast directory rename, not a recompute — a crash during the ~4h
multiband pass itself only costs whatever wasn't written to the temporary path yet, never the
already-complete collection.
</cell id="7cb92f60">

In [ ]:
MIN_POINTS_FOR_MULTIBAND_LS = 10  # combined across all bands, unlike section 1's per-band threshold


def multiband_period_partition(df):
    mb_days = np.full(len(df), np.nan)
    mb_power = np.full(len(df), np.nan)

    for i in range(len(df)):
        row = df.iloc[i]
        t = np.asarray(row["diaSource"]["midpointMjdTai"], dtype=float)
        mag = np.asarray(row["diaSource"]["scienceMag"], dtype=float)
        band = np.asarray(row["diaSource"]["band"])
        sel = np.isfinite(t) & np.isfinite(mag)
        t, mag, band = t[sel], mag[sel], band[sel]

        duration = row["duration_days"]
        max_period = duration / 2 if duration and duration > 0 else np.nan

        if len(t) < MIN_POINTS_FOR_MULTIBAND_LS or not np.isfinite(max_period) or max_period <= 0:
            continue

        freq, power = LombScargleMultiband(t, mag, band).autopower(minimum_frequency=1.0 / max_period, **LS_KWARGS)
        j = np.argmax(power)
        mb_days[i] = float(1.0 / freq[j])
        mb_power[i] = float(power[j])

    df["multiband_period_days"] = mb_days
    df["multiband_period_power"] = mb_power
    return df


if RUN_HEAVY_CALC:
    tmp_path = hq_path.parent / (hq_path.name + "_nb04_multiband_tmp")

    with_multiband_cat = hq_cat.map_partitions(multiband_period_partition)

    t0 = time.time()
    with_multiband_cat.write_catalog(tmp_path, resume=True, create_thumbnail=False, progress_bar=False)
    elapsed = time.time() - t0

    n_obj = hq_cat.hc_structure.catalog_info.total_rows
    print(f"{elapsed:.1f} s for {n_obj} objects ({1000 * elapsed / n_obj:.2f} ms/object)")
    print(f"naive linear projection for the full collection: {FULL_COLLECTION_ROWS * (elapsed / n_obj) / 3600:.0f} hours, serial, single-machine")

    # Promote: the resumable write to a fresh path is complete, so swap it in for the old
    # collection at hq_path — a fast rename, not a recompute.
    if hq_path.exists():
        shutil.rmtree(hq_path)
    tmp_path.rename(hq_path)
    print("promoted", tmp_path, "->", hq_path)

    hq_cat = lsdb.open_catalog(hq_path)
    coverage_df = hq_cat[["multiband_period_days"]].compute()
    n_multiband = coverage_df["multiband_period_days"].notna().sum()
    print(f"multiband periods: {n_multiband} ({100 * n_multiband / n_obj:.1f}%)")

    dp.register(
        name="dia_object_lc_hq",
        type="dp2_subset",
        fmt="hats-collection",
        src_path=str(hq_path / "collection.properties"),
        tags=["nb01", "nb02", "nb03", "nb04", "dp2", "diaObject", "periods", "lomb-scargle", "hq"],
        notes=(
            "nb_01's base HQ sample plus nb_02's stats, nb_03's magnitudes, per-band periods "
            "and best_period_days/best_period_power (highest-power single band) from nb_04 "
            "section 2, plus multiband_period_days/multiband_period_power "
            "(LombScargleMultiband across all bands jointly), from nb_04 section 5."
        ),
        overwrite_history=True,
    )
    dp.print_paths(name="dia_object_lc_hq")
else:
    print(f"RUN_HEAVY_CALC=False — skipping multiband map_partitions/write/register; {hq_path} already has these columns.")
client.close()

Does multiband agree with the single-band `best_period_days` for the ~2400 objects that have
both? Log-log, 1:1 line for reference. Also note `multiband_period_power` stays in `[0, 1]` as
expected for `standard` normalization — section 4's `power=7.93` single-band outlier doesn't
have a multiband counterpart above 1, for whatever that's worth (still not a calibrated FAP).


In [ ]:
# Reopen rather than reuse any in-memory frame from section 5 — safe whether or not
# RUN_HEAVY_CALC just rewrote the collection on disk.
hq_cat = lsdb.open_catalog(hq_path)
compare_cols = ["diaObjectId", "best_period_days", "best_period_power", "multiband_period_days", "multiband_period_power"]
compare_df = hq_cat[compare_cols].compute()

both_df = compare_df.dropna(subset=["best_period_days", "multiband_period_days"])
print(f"n with both: {len(both_df)}")
print(f"multiband_period_power range: [{compare_df['multiband_period_power'].min():.3f}, {compare_df['multiband_period_power'].max():.3f}]")

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(both_df["best_period_days"], both_df["multiband_period_days"], s=6, alpha=0.4, color="C0")
lims = [
    min(both_df["best_period_days"].min(), both_df["multiband_period_days"].min()),
    max(both_df["best_period_days"].max(), both_df["multiband_period_days"].max()),
]
ax.plot(lims, lims, color="gray", linestyle="--", linewidth=1, label="1:1")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("best_period_days (single band)")
ax.set_ylabel("multiband_period_days")
ax.legend(fontsize=8)
fig.tight_layout()

Most points sit in a rough band around the 1:1 line for periods from ~0.3 to ~100 days — some
scatter, but broad agreement. The exception is a distinct group at the bottom left: objects
where the single-band period is very short (`<0.1` day) but multiband lands on something in the
1-4 day range instead — a real disagreement, not just noise around the diagonal. Sub-0.1-day
single-band periods are exactly what a handful of points in one sparse band, with no independent
band to cross-check against, would produce spuriously; multiband pooling more data and landing
somewhere more plausible is a point in its favor, though this isn't a proof either way without
independently vetting a few of these objects (not done here).


## Next

`dia_object_lc_hq` now carries the light curves, nb_02's stats, nb_03's magnitudes, per-band
periods, `best_period_days` (highest-power single band), and `multiband_period_days`
(section 5) together (once `RUN_HEAVY_CALC=True` has been run at least once on this branch) —
one collection, no separate `_with_X` copies.

Open questions carried forward, not resolved here:
- **Not yet run at full HQ scale.** See the heavy-calculation-flag markdown above for the
  extrapolated single-band/multiband time estimates (~3.7 min / ~3.9 hours) — neither has
  actually been measured against the whole ~399k-object sample yet. Both passes are resumable
  (write-to-temp-then-promote), so a crash partway through shouldn't mean starting over — but
  that's only been verified at small scale (see `docs/changelog.md`), not on a real multi-hour
  run.
- **`best_period_power`/`multiband_period_power` aren't false-alarm probabilities.** Single-band
  `power` under `standard` normalization can exceed 1 (section 4's `power=7.93` example folds
  to ~0.05 mag of scatter with no periodic shape — almost certainly noise the periodogram
  overfit). Multiband's power stayed in `[0, 1]` as expected in section 5, which is reassuring
  but still isn't a calibrated significance value. A real vetting pass (bootstrap FAP, or at
  least `false_alarm_probability()`) is needed before treating any of these as candidates.
- **No aliasing/multi-peak check**, single-band or multiband — only the single highest peak is
  kept; the usual daily/yearly-alias companions aren't compared against it.
- **Multiband's coverage/cost tradeoff** (section 5): ~59x slower per object than single-band,
  for going from ~34% to ~99.8% coverage on nb-v01's subset — worth deciding per use case
  whether that tradeoff is worth it, rather than always running both the way this notebook
  does for the demo.
- **`nterms_base`/`nterms_band` left at astropy's defaults (1, 1)** — a simple sinusoid plus
  small per-band offsets. Not explored whether more terms change the picture for this data.
- **`MIN_POINTS_FOR_LS = 10`/`MIN_POINTS_FOR_MULTIBAND_LS = 10` are compromises, not validated
  choices** — see section 1's real-data numbers. Worth revisiting once real candidates need
  vetting rather than a demo.
- **Execution-time projections are naive** (sections 2 and 5) — no parallelism beyond the
  local Dask cluster set up at the top of the notebook, and probably overestimates for the
  true full collection for the reason given in section 2.